# CPU model shootout: gpt-oss-20B vs Gemma-4-26B (run 027)
Serves both GGUFs sequentially on CPU-only Kaggle via
llama-cpp-python server, runs the same 45-IR expansion set per model
(single arm; v1 prompt unified), saves `replies-<model>.jsonl`.
Scoring happens LOCALLY with the pinned TS validator.
Settings: NO accelerator (CPU), Internet ON. ~2-3 h total.


In [ ]:
import subprocess
r = subprocess.run("free -g; nproc", shell=True, capture_output=True, text=True)
print(r.stdout)


In [ ]:
MODELS = [
    ('gptoss', 'unsloth/gpt-oss-20b-GGUF', 'gpt-oss-20b-Q6_K.gguf'),
    ('gemma4', 'unsloth/gemma-4-26B-A4B-it-GGUF',
     'gemma-4-26B-A4B-it-UD-Q6_K.gguf'),
]
N_CTX = 8192
PORT = 8080
TEMP = 0
MAX_TOKENS = 512
print('config OK:', [m[0] for m in MODELS])


In [ ]:
import subprocess
import sys


def pip_install(*args):
    print('+', 'pip install', ' '.join(args), flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args],
                     check=True)


pip_install('--upgrade', 'pip')
# CPU build (no CUDA index): works everywhere, uses all cores.
pip_install('llama-cpp-python')
# Server runtime closure (bare install lacks it; v3/v2 died on sse_starlette).
pip_install('llama-cpp-python[server]')
pip_install('sse-starlette', 'starlette-context', 'fastapi',
            'uvicorn[standard]', 'pydantic', 'pydantic-settings',
            'anyio', 'httpx')
pip_install('huggingface_hub', 'hf_transfer')
import llama_cpp
import sse_starlette, starlette_context, fastapi, uvicorn  # noqa: F401
from llama_cpp.server.app import create_app  # noqa: F401
print('server closure OK (import chain verified pre-download)')
print('llama-cpp-python OK')


In [ ]:
import json as _json
import sys as _sys
import urllib.request as _urlreq

TUNNEL = ""  # stamped by phonehome/push.sh (public receiver URL); empty = disabled
TOKEN = ""  # stamped by phonehome/push.sh (bearer token); empty = disabled
KID = 'cpu-shootout'  # per-kernel short id (cpu-shootout, cpu-lfm25, ...)


def hb(phase, msg=""):
    if not TUNNEL or not TOKEN:
        return
    try:
        body = _json.dumps(
            {"kernel": KID, "phase": phase, "msg": str(msg)[:300]}
        ).encode()
        req = _urlreq.Request(
            f"{TUNNEL}/hb?token={TOKEN}",
            data=body,
            headers={"Content-Type": "application/json"},
        )
        with _urlreq.urlopen(req, timeout=5):
            pass
    except Exception:
        pass


_prev_hook = _sys.excepthook


def _hook(t, v, tb, _p=_prev_hook):
    hb("fatal", f"{t.__name__}: {v}")
    return _p(t, v, tb)


_sys.excepthook = _hook
print("phone-home:", "ON" if TUNNEL and TOKEN else "OFF (unstamped)")

import json
import os
import shutil
import subprocess
import time
import urllib.request

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import hf_hub_download, scan_cache_dir

D = json.load(open('/kaggle/input/datasets/blonded2/qwen-eval-irs/irs.json'))
P = json.load(open('/kaggle/input/datasets/blonded2/qwen-eval-irs/prompts.json'))
assert len(D) == 45, len(D)
PROMPT = P['v0']
assert '{draft}' in PROMPT and 'TESTDRAFT' not in PROMPT
print('eval set OK:', len(D), 'IRs, single arm (v1 unified)')
hb('eval-ok', f'{len(D)} IRs')


def serve_and_wait(gguf_path):
    log = open('/tmp/llama-server.log', 'w')
    cmd = [sys.executable, '-m', 'llama_cpp.server',
           '--model', gguf_path, '--n_gpu_layers', '0',
           '--n_ctx', str(N_CTX), '--host', '127.0.0.1',
           '--port', str(PORT)]
    print('launch:', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    for i in range(60):
        if proc.poll() is not None:
            print('--- server died; log tail ---')
            print(''.join(open('/tmp/llama-server.log').readlines()[-40:]))
            print('--- env capsule ---')
            print(subprocess.run('free -g; df -h /tmp | tail -1', shell=True, capture_output=True, text=True).stdout)
            raise RuntimeError(f"server exited with code {proc.returncode}")
        time.sleep(10)
        try:
            with urllib.request.urlopen(
                    f'http://127.0.0.1:{PORT}/v1/models', timeout=5) as r:
                print('READY, status', r.status, flush=True)
                return proc
        except Exception:
            if i % 6 == 0:
                print(f'waiting for server... ({i * 10}s)', flush=True)
    raise RuntimeError('server did not come up in 10 min')


def chat(prompt):
    body = json.dumps({
        'messages': [{'role': 'user', 'content': prompt}],
        'temperature': TEMP, 'max_tokens': MAX_TOKENS}).encode()
    req = urllib.request.Request(
        f'http://127.0.0.1:{PORT}/v1/chat/completions',
        data=body, headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(req, timeout=900) as r:
        return json.load(r)


for tag, repo, fname in MODELS:
    t0 = time.time()
    print(f'===== {tag}: downloading =====', flush=True)
    hb('downloading', tag)
    path = hf_hub_download(repo_id=repo, filename=fname)
    hb('downloaded', tag)
    proc = serve_and_wait(path)
    hb('ready', tag)
    # smoke: fail fast before burning the eval
    s = chat('Reply with exactly: ok', )
    txt = s['choices'][0]['message']['content'] or ''
    print(f'{tag} smoke:', repr(txt[:80]),
          s['choices'][0]['finish_reason'], flush=True)
    assert txt.strip(), f'{tag} smoke empty'
    hb('smoke', f'{tag} {txt[:60]}')
    outp = f'/kaggle/working/replies-{tag}.jsonl'
    n_err = 0
    toks = []
    with open(outp, 'w') as out:
        for i, item in enumerate(D):
            prompt = PROMPT.replace('{draft}', item['ir'])
            row = {'key': item['key'], 'reply': '', 'finish': '',
                   'error': None, 'wallMs': 0, 'tokPerSec': None, 'compTokens': None}
            ok = False
            for attempt in range(3):
                try:
                    tr0 = time.time()
                    d = chat(prompt)
                    row['reply'] = d['choices'][0]['message']['content'] or ''
                    row['finish'] = d['choices'][0]['finish_reason']
                    row["wallMs"] = round((time.time() - tr0) * 1000)
                    tm = d.get('timings') or {}
                    row['tokPerSec'] = tm.get('predicted_per_second')
                    usg = d.get('usage') or {}
                    row['compTokens'] = usg.get('completion_tokens')
                    toks.append(row["tokPerSec"]) if row["tokPerSec"] else None
                    ok = True
                    break
                except Exception as e:
                    row['error'] = f'{type(e).__name__}: {e}'[:200]
                    time.sleep(10)
            if not ok:
                n_err += 1
            out.write(json.dumps(row) + '\n')
            out.flush()
            if (i + 1) % 10 == 0:
                print(f'{tag}: {i + 1}/45, {n_err} errors' + (f' {sum(toks)/len(toks):.1f}t/s' if toks else ''), flush=True)
                hb('progress', f'{tag} {i + 1}/45 {n_err}err' + (f' {sum(toks)/len(toks):.1f}t/s' if toks else ''))
            assert n_err <= 10, f'error budget exceeded ({n_err})'
    proc.terminate()
    proc.wait(timeout=120)
    # free disk before the next model (server is down, nothing mmapped)
    shutil.rmtree(os.path.expanduser('~/.cache/huggingface'),
                ignore_errors=True)
    print(f'{tag}: done in {(time.time() - t0) / 60:.1f} min', flush=True)
    hb('model-done', f'{tag} {(time.time() - t0) / 60:.1f}min')
print('ALL DONE')
hb('done', 'shootout complete')
